In [ ]:
import os
import gc
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from sklearn.metrics import classification_report, confusion_matrix

from src.utils import (
    set_seed,
    get_score,
    load_yaml_config,
    dict_to_namespace,
    prepare_output_dirs,
)

from src.preprocess import (
    detect_leakage,
    remove_leakage_records,
    apply_text_preprocessing,
)

from src.features import (
    add_basic_text_features,
    validate_targets,
    create_folds,
    get_class_weights,
)

from src.modeling import (
    load_tokenizer,
    run_fold,
)

from src.inference import (
    build_oof_dataframe,
    ensemble_test_predictions,
    build_submission,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 300)
pd.set_option("display.max_columns", 200)

In [ ]:
config_dict = load_yaml_config("configs/config.yaml")
cfg = dict_to_namespace(config_dict)

device = "cuda" if torch.cuda.is_available() else "cpu"

set_seed(cfg.general.seed)
prepare_output_dirs(cfg)

## Data Load

In [ ]:
train = pd.read_csv(cfg.paths.train_csv)
test = pd.read_csv(cfg.paths.test_csv)

print("Original train shape:", train.shape)
print("Original test shape :", test.shape)

display(train.head(3))
display(test.head(3))

## Leakage

In [ ]:
leakage_df, leakage_ids = detect_leakage(
    train,
    text_col=cfg.data.text_col,
    id_col=cfg.data.id_col
)

print("=" * 60)
print("LEAKAGE DETECTION REPORT")
print("=" * 60)
print(f"Total leakage records found: {len(leakage_df)}")
print(f"Unique leakage IDs found   : {leakage_df[cfg.data.id_col].astype(str).nunique()}")

if cfg.data.remove_leakage and len(leakage_ids) > 0:
    train = remove_leakage_records(train, cfg.data.id_col, leakage_ids)

print("\nTrain shape after leakage removal:", train.shape)
print("Removed records:", len(leakage_df))

## Preprocessing & features

In [ ]:
train, test = apply_text_preprocessing(
    train,
    test,
    text_col=cfg.data.text_col,
    use_text_preprocessing=cfg.text.use_text_preprocessing,
    text_version=cfg.text.text_version
)

train = add_basic_text_features(train, cfg.data.text_col)
test = add_basic_text_features(test, cfg.data.text_col)

validate_targets(train, cfg.data.target_col, cfg.data.n_classes)
train = create_folds(train, cfg.data.target_col, cfg.cv.n_folds, cfg.general.seed)

display(pd.crosstab(train["fold"], train[cfg.data.target_col]))

## Tokenizer & class weights

In [ ]:
tokenizer = load_tokenizer(cfg, local_files_only=False)

class_weights = None
if cfg.loss.use_class_weights:
    class_weights = get_class_weights(train, cfg.data.target_col, cfg.data.n_classes)

print("Class weights:", class_weights)

## Folds Training

In [ ]:
all_fold_results = []

for fold in cfg.cv.selected_folds:
    fold_result = run_fold(
        fold=fold,
        train_df=train,
        test_df=test,
        tokenizer=tokenizer,
        cfg=cfg,
        device=device,
        class_weights=class_weights,
        local_files_only=False
    )
    all_fold_results.append(fold_result)

## OOF global

In [ ]:
oof_df = build_oof_dataframe(
    all_fold_results,
    id_col=cfg.data.id_col,
    target_col=cfg.data.target_col,
    n_classes=cfg.data.n_classes
)

cv_score = get_score(oof_df[cfg.data.target_col].values, oof_df["pred"].values)
print("CV Macro F1:", cv_score)

oof_path = os.path.join(cfg.paths.oof_dir, "oof_predictions.csv")
oof_df.to_csv(oof_path, index=False)
print("Saved OOF to:", oof_path)

## Report and Confussion Matrix

In [ ]:
y_true = oof_df[cfg.data.target_col].values
y_pred = oof_df["pred"].values

print(classification_report(y_true, y_pred, digits=4))

cm = confusion_matrix(y_true, y_pred, labels=list(range(cfg.data.n_classes)))

plt.figure(figsize=(7, 6))
plt.imshow(cm, interpolation="nearest")
plt.title("OOF Confusion Matrix")
plt.colorbar()
plt.xticks(range(cfg.data.n_classes), range(cfg.data.n_classes))
plt.yticks(range(cfg.data.n_classes), range(cfg.data.n_classes))
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

## Ensemble and Submission

In [ ]:
test_probs_ens, test_preds = ensemble_test_predictions(
    all_fold_results,
    ensemble_method=cfg.inference.ensemble_method
)

print("Test probs ensemble shape:", test_probs_ens.shape)
print("Test preds shape:", test_preds.shape)

submission_path = os.path.join(cfg.paths.submission_dir, "submission.csv")
submission = build_submission(
    test_df=test,
    id_col=cfg.data.id_col,
    target_col=cfg.data.target_col,
    test_preds=test_preds,
    output_path=submission_path
)

print("Saved organized submission to:", submission_path)
display(submission.head())